# Learn a prior from samples

Flow matching ([Lipman et al., 2023](https://arxiv.org/abs/2210.02747)), on a target that no Gaussian can do: a log-normal process. Positive everywhere, skewed, and those are exactly the two things a Gaussian gets wrong.
```{note}
Tiny on purpose — a couple dozen modes, a couple thousand steps, a minute or two on a CPU. It's
here to show the API, not to be impressive. Real ones live in `FuncyFlows/examples/`.
```

In [ ]:
import matplotlib.pyplot as plt
import torch

from FuncyFlows.base_measures import FourierBasis, GaussianReferenceMeasure
from FuncyFlows.transports.continuous import (ContinuousTransformation, SumField,
                                              LinearField, MatrixField, TimeBasisConditioner)
from FuncyFlows.objectives import FlowMatching
from FuncyFlows.utils.train import train

torch.manual_seed(0)
DTYPE = torch.float64
M, GRID, NUM_TRAIN = 24, 256, 4000

basis = FourierBasis(M, dtype=DTYPE)
points = ((torch.arange(GRID, dtype=DTYPE) + 0.5) / GRID)[:, None]
design = basis.evaluate(points)
project = lambda values: values @ design / GRID

## The data

`f = exp(u)` with `u` a smooth GP of unit pointwise variance. Every single draw is positive. Fit a Gaussian to these and it won't be — that's the test.

In [ ]:
latent = GaussianReferenceMeasure(basis, alpha=0.02, power=2.0, dtype=DTYPE)
u_std = (latent.sample(1000) @ design.T).std()

def simulate(num):
    return project(torch.exp(latent.sample(num) @ design.T / u_std))

train_coeffs = simulate(NUM_TRAIN)
prior_mean = train_coeffs.mean(0)
base = GaussianReferenceMeasure(basis, variances=(train_coeffs - prior_mean).var(0).clamp(min=1e-12))

## The flow

`LinearField` does the per-mode linear part, `MatrixField` adds one wide tanh layer on top. `SumField` glues them together and their traces add, which is why you can stack these without losing exactness. (`MatrixField` is a Sylvester layer — [van den Berg et al., 2018](https://arxiv.org/abs/1803.05649).)

Note `mode_scale=base.scale`. Leave it off and the tanh never sees the high modes at all.

In [ ]:
field = SumField(
    LinearField(M, num_time_modes=4, dtype=DTYPE),
    MatrixField(TimeBasisConditioner(M, 128, num_time_modes=4, dtype=DTYPE),
                mode_scale=base.scale),
)
flow = ContinuousTransformation(base, field, num_steps=12)
print(sum(p.numel() for p in flow.parameters()), "parameters")

In [ ]:
objective = FlowMatching(flow, train_coeffs - prior_mean, batch_size=128,
                         weights=1 / base.scale)
losses = train(objective, flow.parameters(), num_steps=2000, learning_rate=3e-3)
print(f"loss {sum(losses[:50]) / 50:.2f} -> {sum(losses[-50:]) / 50:.2f}")

## Did it learn positivity?

The question a Gaussian can't answer well: what fraction of draws dip below zero *somewhere* on the interval?

In [ ]:
with torch.no_grad():
    flow_draws = flow.transport(base.sample(1000)) + prior_mean

sets = {"data": simulate(1000),
        "diagonal Gaussian": base.sample(1000) + prior_mean,
        "flow": flow_draws}
values = {name: c @ design.T for name, c in sets.items()}
for name, v in values.items():
    print(f"{name:20s} P(min f < 0) = {(v.min(1).values < 0).double().mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 2.8), sharey=True)
for ax, (name, v) in zip(axes, values.items()):
    ax.plot(points[:, 0], v[:20].T, lw=0.8, alpha=0.7)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_title(name, fontsize=10)
plt.show()

## And the density is exact

`log_rn_at` gives you the log RN derivative of the flow's pushforward against the base measure. Closed form — this isn't an estimate, which is the thing that makes the rest of the package possible.

In [ ]:
with torch.no_grad():
    print("log dq/dmu0 on five draws:",
          [round(x, 2) for x in flow.log_rn_at(flow_draws[:5] - prior_mean).tolist()])

---

Next: [a posterior with no training data at all](03_posterior_from_a_likelihood.ipynb).